In [47]:
from pathlib import Path 
from sqlalchemy import create_engine, text 
import pandas as pd 
from string import Template 

db_path = Path("~/Personal Project/data/nba.db").expanduser()
engine = create_engine(f"sqlite:///{db_path}")

In [41]:
with engine.connect() as conn:
    season_teams = pd.read_sql("SELECT Franchise, Year FROM season_teams", engine)
    mappings = pd.read_sql("SELECT * FROM team_abbreviation_to_teamname", engine)

In [43]:
years = season_teams.Year.unique()

In [49]:
url_template = Template("https://www.basketball-reference.com/teams/$abbrev/2025.html")

urls = []
for year in years:
    teams = season_teams[season_teams.Year.eq(year)]
    year_mapping = mappings[mappings.start_year.le(year) & mappings.end_year.ge(year)]

    year_mapping = {tm : abbrev for tm, abbrev in zip(year_mapping["teamname"], year_mapping["abbreviation"])}

    teams["Franchise"] = teams["Franchise"].map(year_mapping)

    urls += [url_template.substitute(abbrev=tm) for tm in teams.Franchise.unique()]
    


In [51]:
len(urls) / 15

114.66666666666667